# Embeddingi — liczby, które rozumieją znaczenie

Na poprzednich zajęciach używaliście modeli do **generowania** tekstu. Ale modele językowe potrafią coś jeszcze — zamieniać tekst w **wektory liczb**.

Taki wektor to właśnie **embedding**. Wygląda tak:

`"pies"      →  [0.23, -0.51,  0.87,  0.12, ..., -0.34]   (768 liczb)`

`"kot"       →  [0.21, -0.48,  0.83,  0.15, ..., -0.31]   (768 liczb)`

`"samochód"  →  [0.84,  0.22, -0.11, -0.67, ...,  0.55]   (768 liczb)`

Kluczowa właściwość: **podobne znaczenie = podobne liczby = bliskie wektory**.

## Jak mierzymy podobieństwo?
Używamy **podobieństwa cosinusowego** — mierzy kąt między dwoma wektorami:
- `1.0` → wektory w tym samym kierunku → bardzo podobne znaczenie
- `0.0` → wektory prostopadłe → zupełnie różne
- `-1.0` → wektory przeciwne → znaczenia przeciwstawne



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

MODEL_NAME = "eryk-mazus/polka-1.1b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

LlamaModel(
  (embed_tokens): Embedding(43904, 2048)
  (layers): ModuleList(
    (0-21): 22 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=256, bias=False)
        (v_proj): Linear(in_features=2048, out_features=256, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
        (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
        (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((2048,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [ ]:
def get_embedding(tekst):
    """
    Zamienia tekst na wektor liczb (embedding).
    Używamy mean pooling: uśredniamy ukryte stany wszystkich tokenów.
    """
    inputs = tokenizer(tekst, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    # Wyciągamy ostatnią warstwę ukrytą i wyliczamy z niej średnią
    outputs = outputs.last_hidden_state.squeeze(0).to(torch.float16)
    return outputs.mean(dim=0).to('cpu').numpy()

def podobienstwo(a, b):
    """Podobieństwo kosinusowe dwóch tekstów. Wynik: od -1.0 do 1.0."""
    ea, eb = get_embedding(a), get_embedding(b)
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

print("Funkcje gotowe!")
print(f"Rozmiar embeddingu dla jednego słowa: {get_embedding('test').shape}")

Funkcje gotowe!
Rozmiar embeddingu dla jednego słowa: (2048,)


---
## Zadanie 1: Kompas znaczeń 🧭

Sprawdzimy, czy embeddingi naprawdę rozumieją znaczenie - czy słowa o podobnym znaczeniu faktycznie mają podobne wektory. Mamy gotową funkcję `podobienstwo(a, b)`, która zwraca liczbę od -1.0 do 1.0 (w praktyce dla słów zwykle od 0.0 do 1.0).

**Zadanie:** Uzupełnij pary słów w tabeli poniżej - dodaj co najmniej 3 własne pary.
Przed uruchomieniem spróbuj zgadnąć, który wynik będzie najwyższy, a który najniższy.

In [ ]:
pary = [
    ("pies",    "kot"),
    ("pies",    "samochód"),
    ("król",    "królowa"),
    ("Polska",  "Warszawa"),
    ("zimno",   "gorąco"),
    # TODO: dopisz tutaj co najmniej 3 własne pary!

]

print(f"{'Para':<30} {'Podobieństwo':>12}")
print("-" * 44)
for a, b in pary:
    s = ...(a, b)  # TODO: jakiej funkcji trzeba użyć?
    pasek = "█" * int(max(0, s) * 20) # max(0, s) zeby nie wywaliło błędu przy ujemnych
    print(f"{a+' vs '+b:<30} {s:>6.3f}  {pasek}")

Para                           Podobieństwo
--------------------------------------------
pies vs kot                     0.702  ██████████████
pies vs samochód                0.674  █████████████
król vs królowa                 0.683  █████████████
Polska vs Warszawa              0.816  ████████████████
zimno vs gorąco                 0.683  █████████████
rower vs hulajnoga              0.601  ████████████
komputer vs klawiatura          0.634  ████████████
słońce vs księżyc               0.726  ██████████████


---
## Zadanie 2: Wyszukiwarka semantyczna 🔍

Klasyczna wyszukiwarka szuka **słów kluczowych** — jeśli wpiszesz "pies", znajdzie tylko dokumenty z dokładnym słowem "pies". Wyszukiwarka semantyczna szuka **znaczenia** — wpiszesz "pies" i znajdzie też zdania o "szczeniaku" albo "czworonogu", bo mają podobne embeddingi.

**Zadanie:** Uzupełnij funkcję `wyszukaj` która:
1. Liczy embedding dla Twojego zapytania
2. Porównuje go z embeddingami wszystkich zdań w bazie
3. Zwraca `n` najbardziej podobnych zdań

In [ ]:
baza = [
    "Pies biega po parku i gania za piłką.",
    "Kot leży na kanapie i śpi przez cały dzień.",
    "Programista pisze kod w Pythonie od rana.",
    "Szczeniak uczył się nowych sztuczek od swojego właściciela.",
    "Komputer zawiesił się podczas kompilowania programu.",
    "Kotek wspina się na drzewo w ogrodzie.",
    "Inżynier debuguje błąd w aplikacji webowej.",
    "Owczarek niemiecki wygrał konkurs na najlepszego psa.",
    "Laptop wymaga aktualizacji systemu operacyjnego.",
    "Rybka pływa w akwarium przy oknie.",
]

# TODO: policz embeddingi całej bazy

embeddingi_bazy = []
...


Liczę embeddingi bazy...
Gotowe!


In [ ]:
def wyszukaj(zapytanie, n=3):
    """Zwraca n zdań z bazy najbardziej podobnych do zapytania."""
    # TODO: Oblicz embedding zapytania używając funkcji get_embedding
    emb_zapytania = ...

    podobienstwa = []
    for i, emb in enumerate(embeddingi_bazy):
        # TODO: Oblicz podobieństwo kosinusowe między emb_zapytania a emb
        # Wskazówka: użyj np.dot() oraz np.linalg.norm()
        s = ...
        podobienstwa.append((s, i))

    # TODO: Posortuj listę `podobienstwa` MALEJĄCO (reverse=True)
    ...

    print(f"Zapytanie: '{zapytanie}'")
    print("-" * 50)
    for s, i in podobienstwa[:n]:
        print(f"  {s:.3f}  {baza[i]}")
    print("\n")

# Po uzupełnieniu kodu, odkomentuj poniższe linie:
# wyszukaj("zwierzę domowe")
# wyszukaj("problemy z komputerem")
# wyszukaj("czworonóg")

Zapytanie: 'zwierzę domowe'
--------------------------------------------------
  0.686  Kotek wspina się na drzewo w ogrodzie.
  0.653  Rybka pływa w akwarium przy oknie.
  0.622  Kot leży na kanapie i śpi przez cały dzień.


Zapytanie: 'problemy z komputerem'
--------------------------------------------------
  0.673  Komputer zawiesił się podczas kompilowania programu.
  0.607  Laptop wymaga aktualizacji systemu operacyjnego.
  0.570  Programista pisze kod w Pythonie od rana.


Zapytanie: 'czworonóg'
--------------------------------------------------
  0.580  Kotek wspina się na drzewo w ogrodzie.
  0.528  Owczarek niemiecki wygrał konkurs na najlepszego psa.
  0.525  Rybka pływa w akwarium przy oknie.




---
## Zadanie 3: Wykrywanie anomalii 🚨

Wyobraź sobie, że masz zbiór zdań, które mówią o tym samym, ale jedno z nich kompletnie nie pasuje. Zamiast czytać wszystko samemu, możemy użyć matematyki! Policzymy tzw. **centroid** (średni wektor dla wszystkich zdań), a potem sprawdzimy, które zdanie jest od niego najdalej.

**Zadanie:** Uzupełnij kod, który znajdzie zdanie niepasujące do tematyki.

In [ ]:
zdania_anomalia = [
    "Złota jesień to najpiękniejsza pora roku.",
    "Liście opadają z drzew w październiku.",
    "Wiosną przyroda budzi się do życia.",
    "Nowy model smartfona ma znacznie lepszy aparat fotograficzny.", # 🚨 To zdanie tu nie pasuje!
    "Zimą często pada śnieg i trzyma mroźna pogoda."
]

# TODO: jakiej funkcji użyjemy do zamiany tekstu na wektor?
emb_anomalia = np.array([...(z) for z in zdania_anomalia])

# TODO: Policz średnią (centroid) ze wszystkich wektorów w emb_anomalia
centroid = ...

odleglosci = []
for i, zdanie in enumerate(zdania_anomalia):
    # TODO: Policz podobieństwo kosinusowe między wektorem zdania a centroidem
    podobieństwo = ...
    odleglosci.append((podobieństwo, zdanie))

# TODO: Posortuj listę tak, aby na początku było zdanie NAJMNIEJ podobne do reszty (najniższy wynik)
...

print("\nNajbardziej odstające zdanie to:")
if len(odleglosci) > 0:
    print(f"🚨 {odleglosci[0][1]} (Wynik podobieństwa: {odleglosci[0][0]:.3f})")

Liczenie embeddingów...

Najbardziej odstające zdanie to:
🚨 Nowy model smartfona ma znacznie lepszy aparat fotograficzny. (Wynik podobieństwa: 0.709)


---
## Zadanie 4: Trening prostej sieci neuronowej 🧠

Skoro umiemy zamieniać tekst na liczby (embeddingi), możemy ich użyć jako danych wejściowych do sieci neuronowej. Zamiast uczyć sieć liter, uczymy ją operować na znaczeniach.

**Zadanie:** Uzupełnij kod, aby wytrenować klasyfikator MLP (Multi-Layer Perceptron), który rozpozna, czy zdanie dotyczy **technologii** czy **natury**.

In [ ]:
import torch.nn as nn
import torch.optim as optim

trening_teksty = [
    "Nowy procesor ma osiem rdzeni.", "Laptop działa bardzo szybko.", "Aplikacja wymaga aktualizacji.",
    "W lesie rosną wysokie sosny.", "Rzeka płynie powoli przez dolinę.", "Jesienią liście zmieniają kolor."
]
y = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)
X = torch.stack([torch.from_numpy(get_embedding(t)).to(torch.float32) for t in trening_teksty])

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Zdefiniuj warstwę liniową (wejście: 2048, wyjście: 2)
        self.fc = ...

    def forward(self, x):
        return self.fc(x)

model_mlp = MyModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.01)

for epoch in range(20):
    optimizer.zero_grad()
    # TODO: Wykonaj forward pass
    outputs = ...
    loss = criterion(outputs, y)

    # TODO: Wykonaj backward pass i krok optymalizacji
    ...

    if (epoch+1) % 5 == 0: print(f"Epoka {epoch+1}, Strata: {loss.item():.4f}")

# 4. Testowanie
test_zdanie = "Mój komputer potrzebuje nowej karty graficznej."
with torch.no_grad():
    emb_test = torch.from_numpy(get_embedding(test_zdanie)).unsqueeze(0).to(torch.float32)
    logity = model_mlp(emb_test)
    # TODO: Jak wyciągnąć numer klasy z największym prawdopodobieństwem?
    wynik = ...

kategorie = {0: "TECHNOLOGIA", 1: "NATURA"}
print(f"\nZdanie: '{test_zdanie}'")
print(f"Wynik: {kategorie[int(wynik)]}")

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. Dane treningowe
trening_tekst = [
    "Nowy procesor ma osiem rdzeni.", "Laptop działa bardzo szybko.", "Aplikacja wymaga aktualizacji.",
    "W lesie rosną wysokie sosny.", "Rzeka płynie powoli przez dolinę.", "Jesienią liście zmieniają kolor."
]
y = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)
X = torch.stack([torch.from_numpy(get_embedding(t)).to(torch.float32) for t in trening_tekst])

# 2. Definicja sieci
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Warstwa liniowa: 2048 cech wejściowych (rozmiar embeddingu) -> 2 klasy wyjściowe
        self.fc = nn.Linear(2048, 2)

    def forward(self, x):
        return self.fc(x)

model_mlp = MyModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mlp.parameters(), lr=0.01)

# 3. Pętla trenowania
print("Trenowanie sieci...")
for epoch in range(20):
    optimizer.zero_grad()
    # Forward pass
    outputs = model_mlp(X)
    loss = criterion(outputs, y)

    # Backward pass i krok optymalizacji
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0: print(f"Epoka {epoch+1}, Strata: {loss.item():.4f}")

# 4. Testowanie
test_zdanie = "Mój komputer potrzebuje nowej karty graficznej."
with torch.no_grad():
    emb_test = torch.from_numpy(get_embedding(test_zdanie)).unsqueeze(0).to(torch.float32)
    logity = model_mlp(emb_test)
    # Wybieramy indeks klasy z najwyższym wynikiem (logitem)
    wynik = torch.argmax(logity, dim=1)

kategorie = {0: "TECHNOLOGIA", 1: "NATURA"}
print(f"\nZdanie: '{test_zdanie}'")
print(f"Wynik: {kategorie[int(wynik)]}")

Trenowanie sieci...
Epoka 5, Strata: 0.0000
Epoka 10, Strata: 0.0000
Epoka 15, Strata: 0.0000
Epoka 20, Strata: 0.0000

Zdanie: 'Mój komputer potrzebuje nowej karty graficznej.'
Wynik: TECHNOLOGIA
